# Protenix in Google Colab: protein-peptide co-folding template

This notebook is a **standalone Protenix implementation for Google Colab**. It is kept as a practical example of how to install Protenix, prepare input JSON files, run predictions, inspect the output tree and export results.

It is **not part of the peptide-screening decision protocol** in this repository. Treat the predicted complexes as exploratory structural hypotheses that require manual inspection and independent validation before they are used for any biological or medicinal-chemistry decision.

Typical use:

```text
protein sequence + peptide CSV
   -> Protenix JSON jobs
   -> optional Protenix execution on Colab GPU
   -> output inspection and ZIP export
```

Recommended Colab setup: `Runtime > Change runtime type > GPU`.

## 0. Install Protenix in an isolated environment

The notebook installs Protenix into `/content/protenix_env` so its ML dependencies do not disturb the Colab notebook kernel. If installation fails because Colab changed CUDA, PyTorch or Python details, restart the runtime and rerun this cell first.

The install can take several minutes. Keep `INSTALL_PROTENIX = True` when you want to run predictions inside Colab. Set it to `False` only if you already have Protenix outputs and just want to inspect/export them.

In [ ]:
%%bash
#@title Install Protenix and lightweight notebook dependencies
set -eu
export DEBIAN_FRONTEND=noninteractive

INSTALL_PROTENIX=true
PROTENIX_ENV=/content/protenix_env

apt-get update -qq || true
apt-get install -y -qq git wget curl zip build-essential ninja-build
python -m pip -q install --upgrade pip wheel packaging virtualenv pandas biopython py3Dmol

if [ "$INSTALL_PROTENIX" = true ]; then
  rm -rf "$PROTENIX_ENV"
  python -m virtualenv "$PROTENIX_ENV"
  "$PROTENIX_ENV/bin/python" -m pip -q install --upgrade pip setuptools wheel ninja
  "$PROTENIX_ENV/bin/python" -m pip install protenix
else
  echo "Skipping Protenix installation. Existing outputs can still be inspected/exported."
fi

python - <<'PY'
from pathlib import Path
import importlib.util, shutil, sys
protenix_bin = Path('/content/protenix_env/bin/protenix')
print('Notebook Python:', sys.version.split()[0])
print('pandas:', 'OK' if importlib.util.find_spec('pandas') else 'missing')
print('Bio:', 'OK' if importlib.util.find_spec('Bio') else 'missing')
print('py3Dmol:', 'OK' if importlib.util.find_spec('py3Dmol') else 'missing')
print('protenix:', protenix_bin if protenix_bin.exists() else shutil.which('protenix') or 'missing')
PY

## 1. Configure the run

Paste the target protein sequence directly, or leave the demo sequence while testing the notebook mechanics. Use only standard one-letter amino-acid codes. The peptide CSV can contain any metadata columns, but it must include `peptide_id` and `sequence`.

Pocket constraints are optional and disabled by default. If you enable them, use **Protenix sequence positions**, not PDB residue numbers.

In [ ]:
#@title Global configuration
from pathlib import Path
import json, os, re, shutil, subprocess, textwrap
import pandas as pd

ROOT = Path('/content/protenix_colab_jobs')
INPUTS = ROOT / 'inputs'
PROTENIX_IN = ROOT / 'protenix_json'
PROTENIX_OUT = ROOT / 'protenix_outputs'
REPORTS = ROOT / 'reports'
for p in [INPUTS, PROTENIX_IN, PROTENIX_OUT, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    # Replace with the complete protein sequence you want to model.
    'target_id': 'target_demo',
    'target_sequence': 'MKWVTFISLLLLFSSAYSRGVFRRDTHKSEIAHRFKDLGE',

    # Protenix run settings. Leave model_name as None unless your installed CLI documents a valid value.
    'seeds': [11, 22, 33],
    'num_samples': 3,
    'model_name': None,
    'use_msa': False,

    # Optional soft pocket guidance. Positions are 1-based sequence positions in the target sequence.
    'use_pocket_constraint': False,
    'pocket_contact_positions': [],
    'pocket_max_distance_A': 8.0,

    # Expected chains in many Protenix outputs: target=A, peptide=B. Verify manually in your files.
    'target_chain_hint': 'A',
    'peptide_chain_hint': 'B',
}

PROTENIX_CMD = str(Path('/content/protenix_env/bin/protenix')) if Path('/content/protenix_env/bin/protenix').exists() else shutil.which('protenix')
if Path('/usr/local/cuda').exists() and not os.environ.get('CUDA_HOME'):
    os.environ['CUDA_HOME'] = '/usr/local/cuda'
    os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ.get('PATH', '')
    os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

print('Workdir:', ROOT)
print('Protenix command:', PROTENIX_CMD or 'missing')
print('CUDA_HOME:', os.environ.get('CUDA_HOME', 'not set'))

## 2. Create or upload peptide inputs

For a real run, replace the example CSV at `/content/protenix_colab_jobs/inputs/peptides.csv` with your own file. Keep peptide IDs short and filesystem-safe.

In [ ]:
#@title Create example peptide CSV
example = pd.DataFrame({
    'peptide_id': ['pep_demo_01', 'pep_demo_02'],
    'sequence': ['LGWVSKGKLL', 'KLVFFAE'],
    'notes': ['demo peptide', 'demo peptide'],
})
peptides_csv = INPUTS / 'peptides.csv'
example.to_csv(peptides_csv, index=False)
print('Edit or replace:', peptides_csv)
example

In [ ]:
#@title Validate peptide table
AA = set('ACDEFGHIKLMNPQRSTVWY')

def clean_sequence(seq):
    return re.sub('[^A-Z]', '', str(seq).upper())

def safe_id(value):
    text = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value).strip())
    return text.strip('._-') or 'peptide'

peptides = pd.read_csv(peptides_csv)
required = {'peptide_id', 'sequence'}
missing = required - set(peptides.columns)
if missing:
    raise ValueError(f'Missing required column(s): {sorted(missing)}')

peptides['peptide_id'] = peptides['peptide_id'].map(safe_id)
peptides['sequence'] = peptides['sequence'].map(clean_sequence)
peptides['valid_natural_aa'] = peptides['sequence'].map(lambda s: bool(s) and set(s).issubset(AA))
peptides['length'] = peptides['sequence'].str.len()
peptides['ready_for_protenix'] = peptides['valid_natural_aa'] & peptides['length'].between(2, 80)

if peptides['peptide_id'].duplicated().any():
    dupes = peptides.loc[peptides['peptide_id'].duplicated(), 'peptide_id'].tolist()
    raise ValueError(f'Duplicate peptide_id values after cleaning: {dupes}')

peptides.to_csv(REPORTS / '01_validated_peptides.csv', index=False)
peptides

## 3. Generate Protenix JSON input files

Each peptide becomes one Protenix job with two protein chains: the target protein and the peptide. The JSON schema used here follows the Protenix input style expected by the `protenix pred` command.

In [ ]:
#@title Generate Protenix input JSON files
def make_protenix_job(peptide_id, peptide_sequence):
    job = {
        'name': peptide_id,
        'sequences': [
            {'proteinChain': {'sequence': CONFIG['target_sequence'], 'count': 1}},
            {'proteinChain': {'sequence': peptide_sequence, 'count': 1}},
        ],
    }
    if CONFIG['use_pocket_constraint']:
        positions = [int(x) for x in CONFIG['pocket_contact_positions']]
        if not positions:
            raise ValueError('Pocket constraints are enabled but pocket_contact_positions is empty.')
        job['constraint'] = {
            'pocket': {
                'binder_chain': {'entity': 2, 'copy': 1},
                'contact_residues': [
                    {'entity': 1, 'copy': 1, 'position': pos}
                    for pos in positions
                ],
                'max_distance': float(CONFIG['pocket_max_distance_A']),
            }
        }
    return [job]

jobs = []
for _, row in peptides[peptides['ready_for_protenix']].iterrows():
    out_json = PROTENIX_IN / f"{row['peptide_id']}.json"
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(make_protenix_job(row['peptide_id'], row['sequence']), f, indent=2)
    jobs.append({'peptide_id': row['peptide_id'], 'sequence': row['sequence'], 'json': str(out_json)})

jobs_df = pd.DataFrame(jobs)
jobs_df.to_csv(REPORTS / '02_protenix_jobs.csv', index=False)
print(f'Wrote {len(jobs_df)} Protenix JSON files to {PROTENIX_IN}')
jobs_df

## 4. Run Protenix predictions

Keep `RUN_PROTENIX = False` until the JSON files look right. When ready, switch it to `True` and run this cell on a GPU runtime.

The command builder checks the installed CLI help and adds only supported options when possible. If your Protenix version uses different argument names, edit `build_protenix_command()` below.

In [ ]:
#@title Run Protenix
RUN_PROTENIX = False #@param {type:"boolean"}

if RUN_PROTENIX and not PROTENIX_CMD:
    raise RuntimeError('RUN_PROTENIX=True but the protenix command is missing. Rerun the install cell first.')
if RUN_PROTENIX and not os.environ.get('CUDA_HOME'):
    print('WARNING: CUDA_HOME is not set. Protenix may fail unless the GPU runtime is active and CUDA is available.')

def run_cmd(cmd, cwd=None, allow_fail=False):
    cmd = [str(x) for x in cmd]
    print('RUN:', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    if result.returncode != 0 and not allow_fail:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {cmd}')
    return result

def protenix_pred_help():
    if not PROTENIX_CMD:
        return ''
    result = subprocess.run([PROTENIX_CMD, 'pred', '--help'], text=True, capture_output=True)
    return (result.stdout or '') + '\n' + (result.stderr or '')

def build_protenix_command(job_json, out_dir, seed):
    help_text = protenix_pred_help()
    cmd = [PROTENIX_CMD, 'pred', '-i', str(job_json), '-o', str(out_dir), '--seeds', str(seed)]
    if '--sample' in help_text:
        cmd += ['--sample', str(CONFIG['num_samples'])]
    elif '--num_samples' in help_text:
        cmd += ['--num_samples', str(CONFIG['num_samples'])]
    else:
        print('WARNING: sample-count option not detected in CLI help; using Protenix default.')
    if CONFIG.get('model_name') and '--model_name' in help_text:
        cmd += ['--model_name', str(CONFIG['model_name'])]
    if not CONFIG.get('use_msa', False) and '--use_msa' in help_text:
        cmd += ['--use_msa', 'false']
    return cmd

if RUN_PROTENIX:
    for job in jobs:
        for seed in CONFIG['seeds']:
            out_dir = PROTENIX_OUT / job['peptide_id'] / f'seed_{seed}'
            out_dir.mkdir(parents=True, exist_ok=True)
            cmd = build_protenix_command(job['json'], out_dir, seed)
            run_cmd(cmd)
else:
    print('RUN_PROTENIX=False. JSON files are ready. Existing outputs should be placed under:', PROTENIX_OUT)

## 5. Inspect output files

This cell inventories the Protenix output folder and records candidate structure files. It does not score or rank peptides.

In [ ]:
#@title Inventory Protenix outputs
import glob

def find_structure_files(base_dir):
    base_dir = Path(base_dir)
    patterns = ['*.cif', '*.mmcif', '*.pdb', '*.cif.gz', '*.mmcif.gz', '*.pdb.gz']
    files = []
    for pattern in patterns:
        files.extend(glob.glob(str(base_dir / '**' / pattern), recursive=True))
    return sorted(files)

rows = []
for job in jobs:
    peptide_dir = PROTENIX_OUT / job['peptide_id']
    all_files = sorted([p for p in peptide_dir.rglob('*') if p.is_file()]) if peptide_dir.exists() else []
    structure_files = find_structure_files(peptide_dir)
    rows.append({
        'peptide_id': job['peptide_id'],
        'output_dir': str(peptide_dir),
        'n_files': len(all_files),
        'n_structure_files': len(structure_files),
        'first_structure_file': structure_files[0] if structure_files else '',
    })
    if all_files:
        print(f"\n{job['peptide_id']} output files:")
        for p in all_files[:30]:
            print(' ', p.relative_to(peptide_dir), p.stat().st_size, 'bytes')
    else:
        print(f"\nNo output files found yet for {job['peptide_id']} under {peptide_dir}")

inventory = pd.DataFrame(rows)
inventory.to_csv(REPORTS / '03_output_inventory.csv', index=False)
inventory

In [ ]:
#@title Optional quick 3D view of the first detected structure
try:
    import py3Dmol
except Exception as e:
    py3Dmol = None
    print('py3Dmol is not available:', e)

first = ''
if 'inventory' in globals() and len(inventory):
    first = inventory.loc[inventory['first_structure_file'].astype(str).str.len() > 0, 'first_structure_file'].head(1).to_list()
    first = first[0] if first else ''

if py3Dmol and first:
    path = Path(first)
    if path.suffix == '.gz':
        import gzip
        data = gzip.open(path, 'rt').read()
        raw_suffix = path.with_suffix('').suffix.lower()
        fmt = 'pdb' if raw_suffix == '.pdb' else 'cif'
    else:
        data = path.read_text(errors='replace')
        fmt = 'pdb' if path.suffix.lower() == '.pdb' else 'cif'
    view = py3Dmol.view(width=900, height=600)
    view.addModel(data, fmt)
    view.setStyle({'chain': CONFIG['target_chain_hint']}, {'cartoon': {'color': 'lightgray'}})
    view.setStyle({'chain': CONFIG['peptide_chain_hint']}, {'cartoon': {'color': 'orange'}, 'stick': {}})
    view.zoomTo()
    view.show()
elif not first:
    print('No structure file found yet. Run Protenix first or copy outputs into PROTENIX_OUT.')

## 6. Export the run folder

The ZIP contains the peptide table, Protenix JSON files, reports and any generated outputs. Download it from the Colab file browser or with `files.download` if desired.

In [ ]:
%%bash
#@title Zip Protenix inputs, reports and outputs
cd /content
zip -qr protenix_colab_jobs.zip protenix_colab_jobs || true
ls -lh /content/protenix_colab_jobs.zip || true